### 1.	Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột.

In [2]:
import os
import findspark

# 1. Cấu hình đường dẫn Java JDK 11 của bạn
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# 2. Kích hoạt kết nối đến thư viện PySpark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, count_distinct, desc, year, month, avg, round, when, datediff

# 3. Khởi tạo Spark Session cô lập, tắt cấu hình Hadoop cũ để tránh lỗi RPC
spark = SparkSession.builder \
    .appName("Lab4_DataFrame_WSL") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Đường dẫn tuyệt đối trỏ thẳng vào thư mục chứa dữ liệu trong WSL Ubuntu của bạn
DATA_PATH = "/home/trananhduy-23520389/BigData/Lab4/" 

# Đọc dữ liệu tự nhận diện kiểu (inferSchema) và dùng dấu phân tách ";"
df_customers = spark.read.csv(DATA_PATH + "Customer_List.csv", header=True, sep=";", inferSchema=True)
df_orders = spark.read.csv(DATA_PATH + "Orders.csv", header=True, sep=";", inferSchema=True)
df_order_items = spark.read.csv(DATA_PATH + "Order_Items.csv", header=True, sep=";", inferSchema=True)
df_reviews = spark.read.csv(DATA_PATH + "Order_Reviews.csv", header=True, sep=";", inferSchema=True)
df_products = spark.read.csv(DATA_PATH + "Products.csv", header=True, sep=";", inferSchema=True)

print("--- BÀI 1: Đã nạp thành công 5 tập dữ liệu DataFrame! ---")
df_orders.printSchema() # Hiển thị cấu trúc bảng Orders để kiểm tra kiểu dữ liệu

your 131072x1 screen size is bogus. expect trouble
26/06/09 12:57:28 WARN Utils: Your hostname, Zuy resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/09 12:57:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 12:57:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


--- BÀI 1: Đã nạp thành công 5 tập dữ liệu DataFrame! ---
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)



### 2.	Thống kê tổng số đơn hàng, số lượng khách hàng và người bán.

In [3]:
# 1. Thống kê tổng số đơn hàng đặt mua (đếm toàn bộ số dòng trong bảng Orders)
total_orders = df_orders.count()

# 2. Thống kê tổng số lượng khách hàng duy nhất (sử dụng dropDuplicates thay cho distinct)
total_customers = df_customers.select("Subscriber_ID").dropDuplicates().count()

# 3. Thống kê tổng số lượng người bán duy nhất (Sellers)
total_sellers = df_order_items.select("Seller_ID").dropDuplicates().count()

# In kết quả định dạng thêm dấu phẩy phân cách phần ngàn cho dễ đọc (ví dụ: 10,000)
print("=========================================")
print("      THỐNG KÊ TỔNG QUAN HỆ THỐNG        ")
print("=========================================")
print(f"Tổng số đơn hàng đặt mua: {total_orders:,}")
print(f"Tổng số lượng khách hàng: {total_customers:,}")
print(f"Tổng số lượng người bán (Sellers): {total_sellers:,}")
print("=========================================")

      THỐNG KÊ TỔNG QUAN HỆ THỐNG        
Tổng số đơn hàng đặt mua: 99,441
Tổng số lượng khách hàng: 99,382
Tổng số lượng người bán (Sellers): 3,095


### 3.	Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần.

In [4]:
# 1. Thực hiện phép Join (kết hợp) giữa bảng Orders (df_orders) và Customer_List (df_customers)
# dựa trên trường chung là 'Customer_Trx_ID' để lấy được thông tin quốc gia của khách hàng
df_orders_with_country = df_orders.join(df_customers, "Customer_Trx_ID")

# 2. Nhóm dữ liệu theo cột quốc gia (Customer_Country), 
# đếm tổng số lượng Order_ID và sắp xếp giảm dần theo số lượng đơn hàng
orders_by_country = df_orders_with_country.groupBy("Customer_Country") \
    .agg(count("Order_ID").alias("Total_Orders")) \
    .orderBy(desc("Total_Orders"))

# 3. Hiển thị kết quả ra bảng thống kê (Không bị cắt chuỗi dữ liệu)
print("=========================================")
print("     SỐ LƯỢNG ĐƠN HÀNG THEO QUỐC GIA     ")
print("=========================================")
orders_by_country.show(truncate=False)

     SỐ LƯỢNG ĐƠN HÀNG THEO QUỐC GIA     
+----------------+------------+
|Customer_Country|Total_Orders|
+----------------+------------+
|Germany         |41754       |
|France          |12848       |
|Netherlands     |11629       |
|Belgium         |5464        |
|Austria         |5043        |
|Switzerland     |3640        |
|United Kingdom  |3382        |
|Poland          |2139        |
|Czechia         |2034        |
|Italy           |2025        |
|Spain           |1651        |
|Portugal        |1336        |
|Sweden          |975         |
|Denmark         |905         |
|Serbia          |746         |
|Norway          |716         |
|Slovakia        |534         |
|Slovenia        |495         |
|Turkey          |485         |
|Greece          |412         |
+----------------+------------+
only showing top 20 rows



### 4.	Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm tăng dần, tháng giảm dần)

In [5]:
# 1. Trích xuất thông tin Năm và Tháng từ cột mốc thời gian mua hàng 'Order_Purchase_Timestamp'
# Hàm year() và month() sẽ tự động bóc tách dữ liệu thời gian ra thành 2 cột mới
df_orders_time = df_orders.withColumn("Year", year("Order_Purchase_Timestamp")) \
                          .withColumn("Month", month("Order_Purchase_Timestamp"))

# 2. Nhóm dữ liệu theo cột 'Year' và 'Month' vừa tạo,
# đếm tổng số lượng đơn hàng (Order_ID), sau đó thực hiện sắp xếp:
# - col("Year").asc(): Năm hiển thị tăng dần (từ cũ đến mới)
# - col("Month").desc(): Tháng hiển thị giảm dần (từ tháng 12 về tháng 1)
orders_by_time = df_orders_time.groupBy("Year", "Month") \
    .agg(count("Order_ID").alias("Total_Orders")) \
    .orderBy(col("Year").asc(), col("Month").desc())

# 3. Hiển thị kết quả thống kê ra bảng
print("=========================================")
print("   SỐ LƯỢNG ĐƠN HÀNG THEO NĂM VÀ THÁNG   ")
print("=========================================")
orders_by_time.show(truncate=False)

   SỐ LƯỢNG ĐƠN HÀNG THEO NĂM VÀ THÁNG   
+----+-----+------------+
|Year|Month|Total_Orders|
+----+-----+------------+
|2022|12   |1           |
|2022|10   |324         |
|2022|9    |4           |
|2023|12   |5673        |
|2023|11   |7544        |
|2023|10   |4631        |
|2023|9    |4285        |
|2023|8    |4331        |
|2023|7    |4026        |
|2023|6    |3245        |
|2023|5    |3700        |
|2023|4    |2404        |
|2023|3    |2682        |
|2023|2    |1780        |
|2023|1    |800         |
|2024|10   |4           |
|2024|9    |16          |
|2024|8    |6512        |
|2024|7    |6292        |
|2024|6    |6167        |
+----+-----+------------+
only showing top 20 rows



### 5.	Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến 5).

In [6]:
# 1. Tiến hành lọc (Filter) làm sạch dữ liệu:
# - col("Review_Score").isNotNull(): Loại bỏ các dòng bị khuyết (NULL)
# - (col("Review_Score") >= 1) & (col("Review_Score") <= 5): Loại bỏ điểm số lỗi/ngoại lệ ngoài thang điểm chuẩn (1 đến 5)
df_reviews_clean = df_reviews.filter(
    col("Review_Score").isNotNull() & 
    (col("Review_Score") >= 1) & 
    (col("Review_Score") <= 5)
)

# 2. Nhóm theo từng mức điểm (Review_Score) để:
# - count("Review_ID"): Đếm tổng số lượng lượt đánh giá ở mức đó
# - round(avg("Review_Score"), 2): Tính điểm trung bình (làm tròn 2 chữ số thập phân, đương nhiên mức 1 trung bình sẽ là 1.0, mức 2 là 2.0...)
# Sau đó sắp xếp tăng dần từ 1 đến 5 theo cột điểm đánh giá
review_stats = df_reviews_clean.groupBy("Review_Score") \
    .agg(
        count("Review_ID").alias("Total_Reviews"),
        round(avg("Review_Score"), 2).alias("Avg_Score")
    ) \
    .orderBy("Review_Score")

# 3. Hiển thị kết quả thống kê ra bảng
print("=========================================")
print("  THỐNG KÊ CHI TIẾT CÁC MỨC ĐIỂM ĐÁNH GIÁ ")
print("=========================================")
review_stats.show(truncate=False)

  THỐNG KÊ CHI TIẾT CÁC MỨC ĐIỂM ĐÁNH GIÁ 
+------------+-------------+---------+
|Review_Score|Total_Reviews|Avg_Score|
+------------+-------------+---------+
|1           |11424        |1.0      |
|2           |3151         |2.0      |
|3           |8179         |3.0      |
|4           |19141        |4.0      |
|5           |57328        |5.0      |
+------------+-------------+---------+



### 6.  Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm

In [7]:
from pyspark.sql.functions import sum as _sum

# 1. Tính doanh thu của từng mặt hàng = Price + Freight_Value
df_items_rev = df_order_items.withColumn("Item_Revenue", col("Price") + col("Freight_Value"))

# 2. Lấy thông tin năm đặt hàng từ bảng Orders (đã xử lý từ Cell 4)
df_orders_2024 = df_orders_time.filter(col("Year") == 2024)

# 3. Join các bảng lại với nhau để lấy Danh mục sản phẩm và tính tổng doanh thu
revenue_by_category = df_items_rev \
    .join(df_orders_2024, "Order_ID") \
    .join(df_products, "Product_ID") \
    .groupBy("Product_Category_Name") \
    .agg(round(_sum("Item_Revenue"), 2).alias("Total_Revenue")) \
    .orderBy(desc("Total_Revenue"))

print("=========================================")
print("    DOANH THU NĂM 2024 THEO DANH MỤC    ")
print("=========================================")
revenue_by_category.show(10, truncate=False)

    DOANH THU NĂM 2024 THEO DANH MỤC    
+---------------------+-------------+
|Product_Category_Name|Total_Revenue|
+---------------------+-------------+
|Health_Beauty        |885191.12    |
|Watches_Gifts        |771986.75    |
|Bed_Bath_Table       |650794.7     |
|Sports_Leisure       |621999.34    |
|Computers_Accessories|594771.04    |
|Housewares           |491576.96    |
|Furniture_Decor      |476466.13    |
|Auto                 |404210.57    |
|Baby                 |299052.56    |
|Cool_Stuff           |273910.05    |
+---------------------+-------------+
only showing top 10 rows



### 7.  Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình cho từng sản phẩm

In [8]:
# 1. Thống kê số lượng bán ra của từng mã sản phẩm
product_sales = df_order_items.groupBy("Product_ID") \
    .agg(count("Order_ID").alias("Units_Sold"))

# 2. Thống kê điểm đánh giá trung bình của từng sản phẩm (dùng df_reviews_clean đã làm sạch ở Cell 5)
product_ratings = df_order_items.join(df_reviews_clean, "Order_ID") \
    .groupBy("Product_ID") \
    .agg(round(avg("Review_Score"), 2).alias("Avg_Rating"))

# 3. Ghép 2 bảng lại và sắp xếp theo số lượng bán ra giảm dần để tìm sản phẩm cao nhất
top_products = product_sales.join(product_ratings, "Product_ID", "left") \
    .orderBy(desc("Units_Sold"))

print("=========================================")
print("       TOP SẢN PHẨM BÁN CHẠY NHẤT        ")
print("=========================================")
top_products.show(10, truncate=False)

       TOP SẢN PHẨM BÁN CHẠY NHẤT        
+--------------------------------+----------+----------+
|Product_ID                      |Units_Sold|Avg_Rating|
+--------------------------------+----------+----------+
|aca2eb7d00ea1a7b8ebd4e68314663af|527       |4.02      |
|99a4788cb24856965c36a24e339b6058|488       |3.9       |
|422879e10f46682990de24d770e7f83d|484       |3.95      |
|389d119b48cf3043d311335e499d9c6b|392       |4.12      |
|368c6c730842d78016ad823897a372db|388       |3.92      |
|53759a2ecddad2bb87a079a1f1519f73|373       |3.87      |
|d1c427060a0f73f6b889a5c7c61f2ac4|343       |4.19      |
|53b36df67ebb7c41585e8d54d6772e08|323       |4.19      |
|154e7e31ebfa092203795c972e5804a6|281       |4.32      |
|3dd2a17168ec895c781a9191c1e95ad7|274       |4.21      |
+--------------------------------+----------+----------+
only showing top 10 rows



### 10. Xếp hạng các seller dựa trên tổng doanh thu và số lượng đơn hàng bán được.

In [9]:
# 1. Nhóm theo Seller_ID từ bảng Order_Items để tính:
# - countDistinct("Order_ID"): Tổng số đơn hàng không trùng lặp mà seller đó nhận được
# - sum("Price"): Tổng doanh thu thuần từ tiền bán sản phẩm
seller_ranking = df_order_items.groupBy("Seller_ID") \
    .agg(
        count_distinct("Order_ID").alias("Total_Orders"),
        round(_sum("Price"), 2).alias("Total_Sales_Revenue")
    ) \
    .orderBy(desc("Total_Sales_Revenue")) # Xếp hạng theo doanh thu cao nhất xuống thấp nhất

print("=========================================")
print("          XẾP HẠNG CÁC SELLER            ")
print("=========================================")
seller_ranking.show(10, truncate=False)

          XẾP HẠNG CÁC SELLER            


+--------------------------------+------------+-------------------+
|Seller_ID                       |Total_Orders|Total_Sales_Revenue|
+--------------------------------+------------+-------------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1132        |229472.63          |
|53243585a1d6dc2643021fd1853d8905|358         |222776.05          |
|4a3ca9315b744ce9f8e9374361493884|1806        |200472.92          |
|fa1c13f2614d7b5c4749cbc52fecda94|585         |194042.03          |
|7c67e1448b00f6e969d365cea6b010ab|982         |187923.89          |
|7e93a43ef30c4f03f38b393420bc753a|336         |176431.87          |
|da8622b14eb17ae2831f4ac5b9dab84a|1314        |160236.57          |
|7a67c85e85bb2ce8582c35f2203ad736|1160        |141745.53          |
|1025f0e2d44d7041d6cf58b6550e0bfa|915         |138968.55          |
|955fee9216a65b617aa5c0531780ce60|1287        |135171.7           |
+--------------------------------+------------+-------------------+
only showing top 10 rows

